In [ ]:
import os
import sys
import subprocess

print("Installing required libraries...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers",
        "librosa",
        "soundfile",
        "scikit-learn",
        "tqdm",
        "joblib"
    ],
    check=True
)

import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import joblib

from tqdm import tqdm
from transformers import AutoFeatureExtractor, Wav2Vec2Model
from torch.utils.data import Dataset, DataLoader

from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

DATASET_ROOT = "/kaggle/input/datasets/ruhinbhattasali/iasnlp-dataset/dataset"

TRAIN_AUDIO_DIR = os.path.join(DATASET_ROOT, "new_train_audio_batches")
VAL_AUDIO_DIR = os.path.join(DATASET_ROOT, "val200_audio_batches")
TEST_AUDIO_DIR = os.path.join(DATASET_ROOT, "test200_audio_batches")

TRAIN_CSV = os.path.join(DATASET_ROOT, "train_dataset_new.csv")
VAL_CSV = os.path.join(DATASET_ROOT, "val_dataset.csv")
TEST_CSV = os.path.join(DATASET_ROOT, "test_dataset.csv")

EMB_DIR = "/kaggle/working/wav2vec2_layer6_mean_pooled_embeddings"
SAVE_DIR = "/kaggle/working/wav2vec2_layer6_sentence_type_outputs"

os.makedirs(EMB_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

print("DATASET_ROOT:", DATASET_ROOT)
print("TRAIN_AUDIO_DIR exists:", os.path.exists(TRAIN_AUDIO_DIR))
print("VAL_AUDIO_DIR exists:", os.path.exists(VAL_AUDIO_DIR))
print("TEST_AUDIO_DIR exists:", os.path.exists(TEST_AUDIO_DIR))
print("TRAIN_CSV exists:", os.path.exists(TRAIN_CSV))
print("VAL_CSV exists:", os.path.exists(VAL_CSV))
print("TEST_CSV exists:", os.path.exists(TEST_CSV))
print("EMB_DIR:", EMB_DIR)
print("SAVE_DIR:", SAVE_DIR)


train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print()
print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)

print()
print("Train columns:", train_df.columns.tolist())
print("Val columns:", val_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())

print()
print("Train sentence_type counts:")
print(train_df["sentence_type"].value_counts())

print()
print("Val label counts:")
print(val_df["label"].value_counts())

print()
print("Test label counts:")
print(test_df["label"].value_counts())

# -------------------------
# Correct label mapping
# -------------------------

TRAIN_LABEL_MAP = {
    "statement": 0,
    "question": 1,
    "verb_start": 2,
    "exclamation": 3
}

VAL_TEST_LABEL_MAP = {
    "declarative": 0,
    "interrogative": 1,
    "imperative": 2,
    "exclamatory": 3
}

LABELS = [
    "declarative",
    "interrogative",
    "imperative",
    "exclamatory"
]

# -------------------------
# Collect audio files
# -------------------------

audio_dirs = [
    TRAIN_AUDIO_DIR,
    VAL_AUDIO_DIR,
    TEST_AUDIO_DIR
]

audio_files = []

print()
print("Counting audio files...")

for audio_dir in audio_dirs:
    count_here = 0

    print()
    print("Checking:", audio_dir)
    print("Folder exists:", os.path.exists(audio_dir))

    if os.path.exists(audio_dir):
        for root, dirs, files in os.walk(audio_dir):
            for f in files:
                if f.lower().endswith((".wav", ".mp3", ".aac", ".flac", ".ogg", ".m4a")):
                    audio_files.append(os.path.join(root, f))
                    count_here += 1

    print("Audio files here:", count_here)

print()
print("Total audio files found:", len(audio_files))

if len(audio_files) == 0:
    raise FileNotFoundError("No audio files found.")

# -------------------------
# Load Wav2Vec2 model
# -------------------------

MODEL_NAME = "facebook/wav2vec2-base-960h"
SR = 16000

# Layer choice
# hidden_states[0] is the feature encoder output before transformer layers
# hidden_states[1] is transformer layer 1
# hidden_states[6] is transformer layer 6
LAYER_ID = 6

device = "cuda" if torch.cuda.is_available() else "cpu"

print()
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Using device:", device)
print("Using Wav2Vec2 hidden layer:", LAYER_ID)

print()
print("Loading Wav2Vec2 feature extractor...")
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)

print("Loading Wav2Vec2 model...")
wav_model = Wav2Vec2Model.from_pretrained(MODEL_NAME)

wav_model = wav_model.to(device)
wav_model.eval()

print("Wav2Vec2 model loaded.")

# -------------------------
# Create layer 6 embeddings
# -------------------------

def make_layer6_embedding(audio_path):
    audio, sr = librosa.load(audio_path, sr=SR)

    inputs = feature_extractor(
        audio,
        sampling_rate=SR,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = wav_model(
            **inputs,
            output_hidden_states=True
        )

    # outputs.hidden_states is a tuple
    # hidden_states[6] means the 6th transformer layer output
    h = outputs.hidden_states[LAYER_ID]

    # h shape: (batch, time_steps, 768)
    # Mean pooling over time gives one vector per audio
    emb = h.mean(dim=1)

    # final shape: (768,)
    emb = emb.squeeze(0).cpu().numpy()

    return emb


print()
print("Creating Wav2Vec2 layer 6 mean-pooled embeddings...")

saved = 0
skipped = 0
already_done = 0

for i, audio_path in enumerate(tqdm(audio_files), start=1):
    fname = os.path.splitext(os.path.basename(audio_path))[0]
    out_path = os.path.join(EMB_DIR, fname + ".npy")

    if os.path.exists(out_path):
        already_done += 1
        continue

    try:
        emb = make_layer6_embedding(audio_path)
        np.save(out_path, emb)
        saved += 1

    except Exception as e:
        skipped += 1
        print("[SKIP]", audio_path, "->", e)

    if i == 1 or i % 100 == 0:
        print(
            "Processed:",
            i,
            "out of",
            len(audio_files),
            "| saved:",
            saved,
            "| already existed:",
            already_done,
            "| skipped:",
            skipped
        )

print()
print("Layer 6 embedding creation done.")
print("Saved new embeddings:", saved)
print("Already existed:", already_done)
print("Skipped:", skipped)

emb_files = []

for root, dirs, files in os.walk(EMB_DIR):
    for f in files:
        if f.lower().endswith(".npy"):
            emb_files.append(os.path.join(root, f))

print("Total layer 6 embedding files:", len(emb_files))

if len(emb_files) > 0:
    sample = np.load(emb_files[0])
    print("Sample embedding shape:", sample.shape)
    print("Example embedding file:", emb_files[0])

# Free GPU memory after embedding extraction
del wav_model
torch.cuda.empty_cache()

# -------------------------
# Build embedding index
# -------------------------

def build_embedding_index(emb_dir):
    file_dict = {}
    duplicate_count = 0

    for root, dirs, files in os.walk(emb_dir):
        for f in files:
            if f.lower().endswith(".npy"):
                key = os.path.splitext(f)[0]
                path = os.path.join(root, f)

                if key in file_dict:
                    duplicate_count += 1

                file_dict[key] = path

    print()
    print("Indexed embeddings:", len(file_dict))
    print("Duplicate base names:", duplicate_count)

    if len(file_dict) == 0:
        raise FileNotFoundError("No embedding files found.")

    return file_dict

# -------------------------
# Load train/val/test embeddings
# -------------------------

def load_train_dataset(df, file_dict):
    X = []
    y = []
    missing = []
    bad_labels = []

    for idx, row in df.iterrows():
        fname = str(row["file_name"])
        label = str(row["sentence_type"]).strip().lower()

        key = os.path.splitext(os.path.basename(fname))[0]

        if label not in TRAIN_LABEL_MAP:
            bad_labels.append(label)
            continue

        if key in file_dict:
            emb = np.load(file_dict[key])
            X.append(emb)
            y.append(TRAIN_LABEL_MAP[label])
        else:
            missing.append(fname)

    X = np.array(X)
    y = np.array(y)

    print()
    print("Training CSV loaded")
    print("Rows:", len(df))
    print("Matched samples:", len(X))
    print("Missing samples:", len(missing))
    print("Bad labels:", len(bad_labels))

    if len(missing) > 0:
        print("First 20 missing:")
        for m in missing[:20]:
            print(m)

    if len(bad_labels) > 0:
        print("Unique bad labels:")
        print(sorted(list(set(bad_labels))))

    if len(X) == 0:
        raise ValueError("No training samples loaded.")

    return X, y


def load_eval_dataset(df, name, file_dict):
    X = []
    y = []
    missing = []
    bad_labels = []

    for idx, row in df.iterrows():
        fname = str(row["file_name"])
        label = str(row["label"]).strip().lower()

        key = os.path.splitext(os.path.basename(fname))[0]

        if label not in VAL_TEST_LABEL_MAP:
            bad_labels.append(label)
            continue

        if key in file_dict:
            emb = np.load(file_dict[key])
            X.append(emb)
            y.append(VAL_TEST_LABEL_MAP[label])
        else:
            missing.append(fname)

    X = np.array(X)
    y = np.array(y)

    print()
    print(name, "CSV loaded")
    print("Rows:", len(df))
    print("Matched samples:", len(X))
    print("Missing samples:", len(missing))
    print("Bad labels:", len(bad_labels))

    if len(missing) > 0:
        print("First 20 missing:")
        for m in missing[:20]:
            print(m)

    if len(bad_labels) > 0:
        print("Unique bad labels:")
        print(sorted(list(set(bad_labels))))

    if len(X) == 0:
        raise ValueError("No samples loaded for " + name)

    return X, y


file_dict = build_embedding_index(EMB_DIR)

X_train, y_train = load_train_dataset(train_df, file_dict)
X_val, y_val = load_eval_dataset(val_df, "Validation", file_dict)
X_test, y_test = load_eval_dataset(test_df, "Test", file_dict)

print()
print("Final shapes:")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

print()
print("Train class distribution:", np.bincount(y_train, minlength=4))
print("Val class distribution:", np.bincount(y_val, minlength=4))
print("Test class distribution:", np.bincount(y_test, minlength=4))

# -------------------------
# Dataset, model, loss
# -------------------------

class LabeledDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X).float()
        self.y = torch.tensor(y).long()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


class ProjectionHead(nn.Module):
    def __init__(self, dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(dim, 512),
            nn.ReLU(),
            nn.Linear(512, 128)
        )

    def forward(self, x):
        z = self.net(x)
        z = F.normalize(z, dim=1)
        return z


def supcon_loss(features, labels, temp=0.5):
    device = features.device

    labels = labels.view(-1, 1)

    mask = torch.eq(labels, labels.T).float().to(device)

    logits = torch.matmul(features, features.T) / temp

    self_mask = torch.eye(len(features), device=device)

    mask = mask * (1 - self_mask)

    exp_logits = torch.exp(logits) * (1 - self_mask)

    log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-9)

    mean_log_prob = (mask * log_prob).sum(1) / (mask.sum(1) + 1e-9)

    loss = -mean_log_prob.mean()

    return loss

# -------------------------
# Train projection model
# -------------------------

NUM_CLASSES = 4
EPOCHS = 80
BATCH_SIZE_LABELED = 64
BATCH_SIZE_PSEUDO = 128
LR = 1e-3
TEMP = 0.5

device = "cuda" if torch.cuda.is_available() else "cpu"

input_dim = X_train.shape[1]

print()
print("Training projection head on device:", device)

proj_model = ProjectionHead(input_dim).to(device)

optimizer = torch.optim.Adam(
    proj_model.parameters(),
    lr=LR
)

for epoch in range(EPOCHS):
    proj_model.train()

    labeled_loader = DataLoader(
        LabeledDataset(X_train, y_train),
        batch_size=BATCH_SIZE_LABELED,
        shuffle=True
    )

    total_loss = 0
    steps = 0

    for x, yb in labeled_loader:
        x = x.to(device)
        yb = yb.to(device)

        z = proj_model(x)

        loss = supcon_loss(z, yb, temp=TEMP)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        steps += 1

    proj_model.eval()

    with torch.no_grad():
        Z_train_all = proj_model(
            torch.tensor(X_train).float().to(device)
        ).cpu().numpy()

    pseudo = KMeans(
        n_clusters=NUM_CLASSES,
        n_init=10,
        random_state=42
    ).fit_predict(Z_train_all)

    pseudo_loader = DataLoader(
        LabeledDataset(X_train, pseudo),
        batch_size=BATCH_SIZE_PSEUDO,
        shuffle=True
    )

    proj_model.train()

    pseudo_loss_total = 0
    pseudo_steps = 0

    for x, yb in pseudo_loader:
        x = x.to(device)
        yb = yb.to(device)

        z = proj_model(x)

        loss = supcon_loss(z, yb, temp=TEMP)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        pseudo_loss_total += loss.item()
        pseudo_steps += 1

    avg_loss = total_loss / max(steps, 1)
    avg_pseudo_loss = pseudo_loss_total / max(pseudo_steps, 1)

    print(
        "Epoch",
        epoch + 1,
        "done | supervised loss:",
        round(avg_loss, 4),
        "| pseudo loss:",
        round(avg_pseudo_loss, 4)
    )

# -------------------------
# Logistic regression evaluation
# -------------------------

def project_embeddings(model, X):
    model.eval()

    with torch.no_grad():
        Z = model(
            torch.tensor(X).float().to(device)
        ).cpu().numpy()

    return Z


def evaluate(name, clf, X_proj, y_true):
    pred = clf.predict(X_proj)

    acc = accuracy_score(y_true, pred)
    prec = precision_score(y_true, pred, average="macro", zero_division=0)
    rec = recall_score(y_true, pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, pred, average="macro", zero_division=0)

    print()
    print("Results on", name)
    print("Accuracy:", acc)
    print("Precision macro:", prec)
    print("Recall macro:", rec)
    print("F1 macro:", f1)

    print()
    print("Classification Report:")
    print(
        classification_report(
            y_true,
            pred,
            target_names=LABELS,
            zero_division=0
        )
    )

    print()
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, pred))

    return pred, acc, f1


print()
print("Projecting embeddings...")

Z_train = project_embeddings(proj_model, X_train)
Z_val = project_embeddings(proj_model, X_val)
Z_test = project_embeddings(proj_model, X_test)

print("Projected train shape:", Z_train.shape)
print("Projected val shape:", Z_val.shape)
print("Projected test shape:", Z_test.shape)

print()
print("Training Logistic Regression classifier...")

clf = LogisticRegression(
    max_iter=3000,
    random_state=42,
    class_weight="balanced"
)

clf.fit(Z_train, y_train)

val_pred, val_acc, val_f1 = evaluate("validation set", clf, Z_val, y_val)
test_pred, test_acc, test_f1 = evaluate("test set", clf, Z_test, y_test)


torch.save(
    proj_model.state_dict(),
    os.path.join(SAVE_DIR, "wav2vec2_layer6_projection_head.pt")
)

joblib.dump(
    clf,
    os.path.join(SAVE_DIR, "layer6_logistic_regression_classifier.joblib")
)

np.save(os.path.join(SAVE_DIR, "layer6_val_predictions.npy"), val_pred)
np.save(os.path.join(SAVE_DIR, "layer6_test_predictions.npy"), test_pred)

summary_path = os.path.join(SAVE_DIR, "layer6_run_summary.txt")

with open(summary_path, "w", encoding="utf-8") as f:
    f.write("Wav2Vec2 layer 6 speech-only sentence type classification\n")
    f.write("\n")
    f.write("Embedding layer used: hidden_states[6]\n")
    f.write("Pooling: mean over time\n")
    f.write("\n")
    f.write("Train mapping:\n")
    f.write("statement -> declarative\n")
    f.write("question -> interrogative\n")
    f.write("verb_start -> imperative\n")
    f.write("exclamation -> exclamatory\n")
    f.write("\n")
    f.write("X_train: " + str(X_train.shape) + "\n")
    f.write("X_val: " + str(X_val.shape) + "\n")
    f.write("X_test: " + str(X_test.shape) + "\n")
    f.write("\n")
    f.write("Validation accuracy: " + str(val_acc) + "\n")
    f.write("Validation macro F1: " + str(val_f1) + "\n")
    f.write("Test accuracy: " + str(test_acc) + "\n")
    f.write("Test macro F1: " + str(test_f1) + "\n")

print()
print("Saved files to:")
print(SAVE_DIR)

print()
print("Saved files:")
for f in os.listdir(SAVE_DIR):
    print(os.path.join(SAVE_DIR, f))

print()
print("All done.")

Installing required libraries...
DATASET_ROOT: /kaggle/input/datasets/ruhinbhattasali/iasnlp-dataset/dataset
TRAIN_AUDIO_DIR exists: True
VAL_AUDIO_DIR exists: True
TEST_AUDIO_DIR exists: True
TRAIN_CSV exists: True
VAL_CSV exists: True
TEST_CSV exists: True
EMB_DIR: /kaggle/working/wav2vec2_layer6_mean_pooled_embeddings
SAVE_DIR: /kaggle/working/wav2vec2_layer6_sentence_type_outputs

Train shape: (4000, 4)
Val shape: (202, 3)
Test shape: (202, 3)

Train columns: ['file_name', 'transcript', 'label', 'sentence_type']
Val columns: ['file_name', 'transcript', 'label']
Test columns: ['file_name', 'transcript', 'label']

Train sentence_type counts:
sentence_type
verb_start     1100
statement      1100
question       1100
exclamation     700
Name: count, dtype: int64

Val label counts:
label
declarative      53
exclamatory      51
interrogative    49
imperative       49
Name: count, dtype: int64

Test label counts:
label
declarative      53
exclamatory      51
imperative       50
interrogati

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Wav2Vec2 model loaded.

Creating Wav2Vec2 layer 6 mean-pooled embeddings...


100%|██████████| 4404/4404 [00:00<00:00, 109976.87it/s]


Layer 6 embedding creation done.
Saved new embeddings: 0
Already existed: 4404
Skipped: 0
Total layer 6 embedding files: 4404
Sample embedding shape: (768,)
Example embedding file: /kaggle/working/wav2vec2_layer6_mean_pooled_embeddings/4_0_d1305.npy

Indexed embeddings: 4404
Duplicate base names: 0



Training CSV loaded
Rows: 4000
Matched samples: 4000
Missing samples: 0
Bad labels: 0

Validation CSV loaded
Rows: 202
Matched samples: 202
Missing samples: 0
Bad labels: 0

Test CSV loaded
Rows: 202
Matched samples: 202
Missing samples: 0
Bad labels: 0

Final shapes:
X_train: (4000, 768) y_train: (4000,)
X_val: (202, 768) y_val: (202,)
X_test: (202, 768) y_test: (202,)

Train class distribution: [1100 1100 1100  700]
Val class distribution: [53 49 49 51]
Test class distribution: [53 48 50 51]

Training projection head on device: cuda
Epoch 1 done | supervised loss: 4.0084 | pseudo loss: 4.0762
Epoch 2 done | supervised loss: 3.8987 | pseudo loss: 4.0261
Epoch 3 done | supervised loss: 3.8225 | pseudo loss: 4.0076
Epoch 4 done | supervised loss: 3.7639 | pseudo loss: 3.994
Epoch 5 done | supervised loss: 3.7131 | pseudo loss: 3.9852
Epoch 6 done | supervised loss: 3.6202 | pseudo loss: 3.9349
Epoch 7 done | supervised loss: 3.547 | pseudo loss: 3.9353
Epoch 8 done | supervised loss: 3

In [11]:
import os
import shutil
from IPython.display import FileLink, display

OUTPUT_DIR = "/kaggle/working/wav2vec2_layer6_sentence_type_outputs"

# Move to /kaggle/working so the link becomes relative
os.chdir("/kaggle/working")

ZIP_NAME = "wav2vec2_layer6_sentence_type_outputs.zip"
ZIP_BASE = "wav2vec2_layer6_sentence_type_outputs"

# Remove old zip if it exists
if os.path.exists(ZIP_NAME):
    os.remove(ZIP_NAME)

# Create zip
shutil.make_archive(
    ZIP_BASE,
    "zip",
    OUTPUT_DIR
)

print("Zip created at:")
print("/kaggle/working/" + ZIP_NAME)

print()
print("File size:")
!ls -lh /kaggle/working/wav2vec2_layer6_sentence_type_outputs.zip

display(FileLink(ZIP_NAME))

Zip created at:
/kaggle/working/wav2vec2_layer6_sentence_type_outputs.zip

File size:
-rw-r--r-- 1 root root 1.7M Jun 25 18:07 /kaggle/working/wav2vec2_layer6_sentence_type_outputs.zip


/kaggle/working/wav2vec2_layer6_sentence_type_outputs.zip

In [ ]:
import os
import sys
import subprocess

print("Installing required libraries...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers",
        "librosa",
        "soundfile",
        "scikit-learn",
        "tqdm",
        "joblib"
    ],
    check=True
)

import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import joblib
import shutil

from tqdm import tqdm
from IPython.display import FileLink, display
from transformers import AutoFeatureExtractor, Wav2Vec2Model
from torch.utils.data import Dataset, DataLoader

from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

DATASET_ROOT = "/kaggle/input/datasets/ruhinbhattasali/iasnlp-dataset/dataset"

TRAIN_AUDIO_DIR = os.path.join(DATASET_ROOT, "new_train_audio_batches")
VAL_AUDIO_DIR = os.path.join(DATASET_ROOT, "val200_audio_batches")
TEST_AUDIO_DIR = os.path.join(DATASET_ROOT, "test200_audio_batches")

TRAIN_CSV = os.path.join(DATASET_ROOT, "train_dataset_new.csv")
VAL_CSV = os.path.join(DATASET_ROOT, "val_dataset.csv")
TEST_CSV = os.path.join(DATASET_ROOT, "test_dataset.csv")

# New folders for layer 9
EMB_DIR = "/kaggle/working/wav2vec2_layer9_mean_pooled_embeddings"
SAVE_DIR = "/kaggle/working/wav2vec2_layer9_sentence_type_outputs"

os.makedirs(EMB_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

print("DATASET_ROOT:", DATASET_ROOT)
print("TRAIN_AUDIO_DIR exists:", os.path.exists(TRAIN_AUDIO_DIR))
print("VAL_AUDIO_DIR exists:", os.path.exists(VAL_AUDIO_DIR))
print("TEST_AUDIO_DIR exists:", os.path.exists(TEST_AUDIO_DIR))
print("TRAIN_CSV exists:", os.path.exists(TRAIN_CSV))
print("VAL_CSV exists:", os.path.exists(VAL_CSV))
print("TEST_CSV exists:", os.path.exists(TEST_CSV))
print("EMB_DIR:", EMB_DIR)
print("SAVE_DIR:", SAVE_DIR)

# -------------------------
# Read CSV files
# -------------------------

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print()
print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)

print()
print("Train sentence_type counts:")
print(train_df["sentence_type"].value_counts())

print()
print("Val label counts:")
print(val_df["label"].value_counts())

print()
print("Test label counts:")
print(test_df["label"].value_counts())

# -------------------------
# Correct label mapping
# -------------------------
# train sentence_type:
# statement -> declarative
# question -> interrogative
# verb_start -> imperative
# exclamation -> exclamatory
#
# This matches the train/val/test label structure in your Kaggle output. 
# Val/test already use declarative/interrogative/imperative/exclamatory.

TRAIN_LABEL_MAP = {
    "statement": 0,
    "question": 1,
    "verb_start": 2,
    "exclamation": 3
}

VAL_TEST_LABEL_MAP = {
    "declarative": 0,
    "interrogative": 1,
    "imperative": 2,
    "exclamatory": 3
}

LABELS = [
    "declarative",
    "interrogative",
    "imperative",
    "exclamatory"
]

# -------------------------
# Collect audio files
# -------------------------

audio_dirs = [
    TRAIN_AUDIO_DIR,
    VAL_AUDIO_DIR,
    TEST_AUDIO_DIR
]

audio_files = []

print()
print("Counting audio files...")

for audio_dir in audio_dirs:
    count_here = 0

    print()
    print("Checking:", audio_dir)
    print("Folder exists:", os.path.exists(audio_dir))

    if os.path.exists(audio_dir):
        for root, dirs, files in os.walk(audio_dir):
            for f in files:
                if f.lower().endswith((".wav", ".mp3", ".aac", ".flac", ".ogg", ".m4a")):
                    audio_files.append(os.path.join(root, f))
                    count_here += 1

    print("Audio files here:", count_here)

print()
print("Total audio files found:", len(audio_files))

if len(audio_files) == 0:
    raise FileNotFoundError("No audio files found.")

# -------------------------
# Load Wav2Vec2 model
# -------------------------

MODEL_NAME = "facebook/wav2vec2-base-960h"
SR = 16000

# Important:
# hidden_states[0] = feature encoder output before transformer layers
# hidden_states[1] = transformer layer 1
# hidden_states[9] = transformer layer 9
LAYER_ID = 9

device = "cuda" if torch.cuda.is_available() else "cpu"

print()
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Using device:", device)
print("Using Wav2Vec2 hidden layer:", LAYER_ID)

print()
print("Loading Wav2Vec2 feature extractor...")
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)

print("Loading Wav2Vec2 model...")
wav_model = Wav2Vec2Model.from_pretrained(MODEL_NAME)

wav_model = wav_model.to(device)
wav_model.eval()

print("Wav2Vec2 model loaded.")

# -------------------------
# Create layer 9 embeddings
# -------------------------

def make_layer9_embedding(audio_path):
    audio, sr = librosa.load(audio_path, sr=SR)

    inputs = feature_extractor(
        audio,
        sampling_rate=SR,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = wav_model(
            **inputs,
            output_hidden_states=True
        )

    # Layer 9 output
    h = outputs.hidden_states[LAYER_ID]

    # Mean pooling over time
    emb = h.mean(dim=1)

    # Shape becomes (768,)
    emb = emb.squeeze(0).cpu().numpy()

    return emb


print()
print("Creating Wav2Vec2 layer 9 mean-pooled embeddings...")

saved = 0
skipped = 0
already_done = 0

for i, audio_path in enumerate(tqdm(audio_files), start=1):
    fname = os.path.splitext(os.path.basename(audio_path))[0]
    out_path = os.path.join(EMB_DIR, fname + ".npy")

    if os.path.exists(out_path):
        already_done += 1
        continue

    try:
        emb = make_layer9_embedding(audio_path)
        np.save(out_path, emb)
        saved += 1

    except Exception as e:
        skipped += 1
        print("[SKIP]", audio_path, "->", e)

    if i == 1 or i % 100 == 0:
        print(
            "Processed:",
            i,
            "out of",
            len(audio_files),
            "| saved:",
            saved,
            "| already existed:",
            already_done,
            "| skipped:",
            skipped
        )

print()
print("Layer 9 embedding creation done.")
print("Saved new embeddings:", saved)
print("Already existed:", already_done)
print("Skipped:", skipped)

emb_files = []

for root, dirs, files in os.walk(EMB_DIR):
    for f in files:
        if f.lower().endswith(".npy"):
            emb_files.append(os.path.join(root, f))

print("Total layer 9 embedding files:", len(emb_files))

if len(emb_files) > 0:
    sample = np.load(emb_files[0])
    print("Sample embedding shape:", sample.shape)
    print("Example embedding file:", emb_files[0])

# Free GPU memory after embedding extraction
del wav_model
torch.cuda.empty_cache()

# -------------------------
# Build embedding index
# -------------------------

def build_embedding_index(emb_dir):
    file_dict = {}
    duplicate_count = 0

    for root, dirs, files in os.walk(emb_dir):
        for f in files:
            if f.lower().endswith(".npy"):
                key = os.path.splitext(f)[0]
                path = os.path.join(root, f)

                if key in file_dict:
                    duplicate_count += 1

                file_dict[key] = path

    print()
    print("Indexed embeddings:", len(file_dict))
    print("Duplicate base names:", duplicate_count)

    if len(file_dict) == 0:
        raise FileNotFoundError("No embedding files found.")

    return file_dict

# -------------------------
# Load train/val/test embeddings
# -------------------------

def load_train_dataset(df, file_dict):
    X = []
    y = []
    missing = []
    bad_labels = []

    for idx, row in df.iterrows():
        fname = str(row["file_name"])
        label = str(row["sentence_type"]).strip().lower()

        key = os.path.splitext(os.path.basename(fname))[0]

        if label not in TRAIN_LABEL_MAP:
            bad_labels.append(label)
            continue

        if key in file_dict:
            emb = np.load(file_dict[key])
            X.append(emb)
            y.append(TRAIN_LABEL_MAP[label])
        else:
            missing.append(fname)

    X = np.array(X)
    y = np.array(y)

    print()
    print("Training CSV loaded")
    print("Rows:", len(df))
    print("Matched samples:", len(X))
    print("Missing samples:", len(missing))
    print("Bad labels:", len(bad_labels))

    if len(missing) > 0:
        print("First 20 missing:")
        for m in missing[:20]:
            print(m)

    if len(bad_labels) > 0:
        print("Unique bad labels:")
        print(sorted(list(set(bad_labels))))

    if len(X) == 0:
        raise ValueError("No training samples loaded.")

    return X, y


def load_eval_dataset(df, name, file_dict):
    X = []
    y = []
    missing = []
    bad_labels = []

    for idx, row in df.iterrows():
        fname = str(row["file_name"])
        label = str(row["label"]).strip().lower()

        key = os.path.splitext(os.path.basename(fname))[0]

        if label not in VAL_TEST_LABEL_MAP:
            bad_labels.append(label)
            continue

        if key in file_dict:
            emb = np.load(file_dict[key])
            X.append(emb)
            y.append(VAL_TEST_LABEL_MAP[label])
        else:
            missing.append(fname)

    X = np.array(X)
    y = np.array(y)

    print()
    print(name, "CSV loaded")
    print("Rows:", len(df))
    print("Matched samples:", len(X))
    print("Missing samples:", len(missing))
    print("Bad labels:", len(bad_labels))

    if len(missing) > 0:
        print("First 20 missing:")
        for m in missing[:20]:
            print(m)

    if len(bad_labels) > 0:
        print("Unique bad labels:")
        print(sorted(list(set(bad_labels))))

    if len(X) == 0:
        raise ValueError("No samples loaded for " + name)

    return X, y


file_dict = build_embedding_index(EMB_DIR)

X_train, y_train = load_train_dataset(train_df, file_dict)
X_val, y_val = load_eval_dataset(val_df, "Validation", file_dict)
X_test, y_test = load_eval_dataset(test_df, "Test", file_dict)

print()
print("Final shapes:")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

print()
print("Train class distribution:", np.bincount(y_train, minlength=4))
print("Val class distribution:", np.bincount(y_val, minlength=4))
print("Test class distribution:", np.bincount(y_test, minlength=4))

# -------------------------
# Dataset, model, loss
# -------------------------

class LabeledDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X).float()
        self.y = torch.tensor(y).long()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


class ProjectionHead(nn.Module):
    def __init__(self, dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(dim, 512),
            nn.ReLU(),
            nn.Linear(512, 128)
        )

    def forward(self, x):
        z = self.net(x)
        z = F.normalize(z, dim=1)
        return z


def supcon_loss(features, labels, temp=0.5):
    device = features.device

    labels = labels.view(-1, 1)

    mask = torch.eq(labels, labels.T).float().to(device)

    logits = torch.matmul(features, features.T) / temp

    self_mask = torch.eye(len(features), device=device)

    mask = mask * (1 - self_mask)

    exp_logits = torch.exp(logits) * (1 - self_mask)

    log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-9)

    mean_log_prob = (mask * log_prob).sum(1) / (mask.sum(1) + 1e-9)

    loss = -mean_log_prob.mean()

    return loss

# -------------------------
# Train projection model
# -------------------------

NUM_CLASSES = 4
EPOCHS = 80
BATCH_SIZE_LABELED = 64
BATCH_SIZE_PSEUDO = 128
LR = 1e-3
TEMP = 0.5

device = "cuda" if torch.cuda.is_available() else "cpu"

input_dim = X_train.shape[1]

print()
print("Training projection head on device:", device)

proj_model = ProjectionHead(input_dim).to(device)

optimizer = torch.optim.Adam(
    proj_model.parameters(),
    lr=LR
)

for epoch in range(EPOCHS):
    proj_model.train()

    labeled_loader = DataLoader(
        LabeledDataset(X_train, y_train),
        batch_size=BATCH_SIZE_LABELED,
        shuffle=True
    )

    total_loss = 0
    steps = 0

    for x, yb in labeled_loader:
        x = x.to(device)
        yb = yb.to(device)

        z = proj_model(x)

        loss = supcon_loss(z, yb, temp=TEMP)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        steps += 1

    proj_model.eval()

    with torch.no_grad():
        Z_train_all = proj_model(
            torch.tensor(X_train).float().to(device)
        ).cpu().numpy()

    pseudo = KMeans(
        n_clusters=NUM_CLASSES,
        n_init=10,
        random_state=42
    ).fit_predict(Z_train_all)

    pseudo_loader = DataLoader(
        LabeledDataset(X_train, pseudo),
        batch_size=BATCH_SIZE_PSEUDO,
        shuffle=True
    )

    proj_model.train()

    pseudo_loss_total = 0
    pseudo_steps = 0

    for x, yb in pseudo_loader:
        x = x.to(device)
        yb = yb.to(device)

        z = proj_model(x)

        loss = supcon_loss(z, yb, temp=TEMP)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        pseudo_loss_total += loss.item()
        pseudo_steps += 1

    avg_loss = total_loss / max(steps, 1)
    avg_pseudo_loss = pseudo_loss_total / max(pseudo_steps, 1)

    print(
        "Epoch",
        epoch + 1,
        "done | supervised loss:",
        round(avg_loss, 4),
        "| pseudo loss:",
        round(avg_pseudo_loss, 4)
    )

# -------------------------
# Logistic regression evaluation
# -------------------------

def project_embeddings(model, X):
    model.eval()

    with torch.no_grad():
        Z = model(
            torch.tensor(X).float().to(device)
        ).cpu().numpy()

    return Z


def evaluate(name, clf, X_proj, y_true):
    pred = clf.predict(X_proj)

    acc = accuracy_score(y_true, pred)
    prec = precision_score(y_true, pred, average="macro", zero_division=0)
    rec = recall_score(y_true, pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, pred, average="macro", zero_division=0)

    print()
    print("Results on", name)
    print("Accuracy:", acc)
    print("Precision macro:", prec)
    print("Recall macro:", rec)
    print("F1 macro:", f1)

    print()
    print("Classification Report:")
    print(
        classification_report(
            y_true,
            pred,
            target_names=LABELS,
            zero_division=0
        )
    )

    print()
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, pred))

    return pred, acc, f1


print()
print("Projecting embeddings...")

Z_train = project_embeddings(proj_model, X_train)
Z_val = project_embeddings(proj_model, X_val)
Z_test = project_embeddings(proj_model, X_test)

print("Projected train shape:", Z_train.shape)
print("Projected val shape:", Z_val.shape)
print("Projected test shape:", Z_test.shape)

print()
print("Training Logistic Regression classifier...")

clf = LogisticRegression(
    max_iter=3000,
    random_state=42,
    class_weight="balanced"
)

clf.fit(Z_train, y_train)

val_pred, val_acc, val_f1 = evaluate("validation set", clf, Z_val, y_val)
test_pred, test_acc, test_f1 = evaluate("test set", clf, Z_test, y_test)

# -------------------------
# Save outputs
# -------------------------

torch.save(
    proj_model.state_dict(),
    os.path.join(SAVE_DIR, "wav2vec2_layer9_projection_head.pt")
)

joblib.dump(
    clf,
    os.path.join(SAVE_DIR, "layer9_logistic_regression_classifier.joblib")
)

np.save(os.path.join(SAVE_DIR, "layer9_val_predictions.npy"), val_pred)
np.save(os.path.join(SAVE_DIR, "layer9_test_predictions.npy"), test_pred)

summary_path = os.path.join(SAVE_DIR, "layer9_run_summary.txt")

with open(summary_path, "w", encoding="utf-8") as f:
    f.write("Wav2Vec2 layer 9 speech-only sentence type classification\n")
    f.write("\n")
    f.write("Embedding layer used: hidden_states[9]\n")
    f.write("Pooling: mean over time\n")
    f.write("\n")
    f.write("Train mapping:\n")
    f.write("statement -> declarative\n")
    f.write("question -> interrogative\n")
    f.write("verb_start -> imperative\n")
    f.write("exclamation -> exclamatory\n")
    f.write("\n")
    f.write("X_train: " + str(X_train.shape) + "\n")
    f.write("X_val: " + str(X_val.shape) + "\n")
    f.write("X_test: " + str(X_test.shape) + "\n")
    f.write("\n")
    f.write("Validation accuracy: " + str(val_acc) + "\n")
    f.write("Validation macro F1: " + str(val_f1) + "\n")
    f.write("Test accuracy: " + str(test_acc) + "\n")
    f.write("Test macro F1: " + str(test_f1) + "\n")

print()
print("Saved files to:")
print(SAVE_DIR)

print()
print("Saved files:")
for f in os.listdir(SAVE_DIR):
    print(os.path.join(SAVE_DIR, f))


os.chdir("/kaggle/working")

ZIP_NAME = "wav2vec2_layer9_sentence_type_outputs.zip"
ZIP_BASE = "wav2vec2_layer9_sentence_type_outputs"

if os.path.exists(ZIP_NAME):
    os.remove(ZIP_NAME)

shutil.make_archive(
    ZIP_BASE,
    "zip",
    SAVE_DIR
)

print()
print("Zip created:")
print("/kaggle/working/" + ZIP_NAME)

print()
print("Zip size:")
!ls -lh /kaggle/working/wav2vec2_layer9_sentence_type_outputs.zip

display(FileLink(ZIP_NAME))

print()
print("All done.")

Installing required libraries...
DATASET_ROOT: /kaggle/input/datasets/ruhinbhattasali/iasnlp-dataset/dataset
TRAIN_AUDIO_DIR exists: True
VAL_AUDIO_DIR exists: True
TEST_AUDIO_DIR exists: True
TRAIN_CSV exists: True
VAL_CSV exists: True
TEST_CSV exists: True
EMB_DIR: /kaggle/working/wav2vec2_layer9_mean_pooled_embeddings
SAVE_DIR: /kaggle/working/wav2vec2_layer9_sentence_type_outputs

Train shape: (4000, 4)
Val shape: (202, 3)
Test shape: (202, 3)

Train sentence_type counts:
sentence_type
verb_start     1100
statement      1100
question       1100
exclamation     700
Name: count, dtype: int64

Val label counts:
label
declarative      53
exclamatory      51
interrogative    49
imperative       49
Name: count, dtype: int64

Test label counts:
label
declarative      53
exclamatory      51
imperative       50
interrogative    48
Name: count, dtype: int64

Counting audio files...

Checking: /kaggle/input/datasets/ruhinbhattasali/iasnlp-dataset/dataset/new_train_audio_batches
Folder exists:

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Wav2Vec2 model loaded.

Creating Wav2Vec2 layer 9 mean-pooled embeddings...


  0%|          | 3/4404 [00:00<02:43, 26.88it/s]

...

Processed: 4300 out of 4404 | saved: 4300 | already existed: 0 | skipped: 0


100%|██████████| 4404/4404 [02:01<00:00, 36.37it/s]


Processed: 4400 out of 4404 | saved: 4400 | already existed: 0 | skipped: 0

Layer 9 embedding creation done.
Saved new embeddings: 4404
Already existed: 0
Skipped: 0
Total layer 9 embedding files: 4404
Sample embedding shape: (768,)
Example embedding file: /kaggle/working/wav2vec2_layer9_mean_pooled_embeddings/4_0_d1305.npy

Indexed embeddings: 4404
Duplicate base names: 0

Training CSV loaded
Rows: 4000
Matched samples: 4000
Missing samples: 0
Bad labels: 0

Validation CSV loaded
Rows: 202
Matched samples: 202
Missing samples: 0
Bad labels: 0

Test CSV loaded
Rows: 202
Matched samples: 202
Missing samples: 0
Bad labels: 0

Final shapes:
X_train: (4000, 768) y_train: (4000,)
X_val: (202, 768) y_val: (202,)
X_test: (202, 768) y_test: (202,)

Train class distribution: [1100 1100 1100  700]
Val class distribution: [53 49 49 51]
Test class distribution: [53 48 50 51]

Training projection head on device: cuda
Epoch 1 done | supervised loss: 4.038 | pseudo loss: 4.203
Epoch 2 done | supervi

/kaggle/working/wav2vec2_layer9_sentence_type_outputs.zip


All done.


In [ ]:
import os
import sys
import subprocess

print("Installing required libraries...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers",
        "librosa",
        "soundfile",
        "scikit-learn",
        "tqdm",
        "joblib"
    ],
    check=True
)

import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import joblib
import shutil

from tqdm import tqdm
from IPython.display import FileLink, display
from transformers import AutoFeatureExtractor, Wav2Vec2Model
from torch.utils.data import Dataset, DataLoader

from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

DATASET_ROOT = "/kaggle/input/datasets/ruhinbhattasali/iasnlp-dataset/dataset"

TRAIN_AUDIO_DIR = os.path.join(DATASET_ROOT, "new_train_audio_batches")
VAL_AUDIO_DIR = os.path.join(DATASET_ROOT, "val200_audio_batches")
TEST_AUDIO_DIR = os.path.join(DATASET_ROOT, "test200_audio_batches")

TRAIN_CSV = os.path.join(DATASET_ROOT, "train_dataset_new.csv")
VAL_CSV = os.path.join(DATASET_ROOT, "val_dataset.csv")
TEST_CSV = os.path.join(DATASET_ROOT, "test_dataset.csv")

EMB_DIR = "/kaggle/working/wav2vec2_layer7_mean_pooled_embeddings"
SAVE_DIR = "/kaggle/working/wav2vec2_layer7_sentence_type_outputs"

os.makedirs(EMB_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

print("DATASET_ROOT:", DATASET_ROOT)
print("TRAIN_AUDIO_DIR exists:", os.path.exists(TRAIN_AUDIO_DIR))
print("VAL_AUDIO_DIR exists:", os.path.exists(VAL_AUDIO_DIR))
print("TEST_AUDIO_DIR exists:", os.path.exists(TEST_AUDIO_DIR))
print("TRAIN_CSV exists:", os.path.exists(TRAIN_CSV))
print("VAL_CSV exists:", os.path.exists(VAL_CSV))
print("TEST_CSV exists:", os.path.exists(TEST_CSV))
print("EMB_DIR:", EMB_DIR)
print("SAVE_DIR:", SAVE_DIR)

# -------------------------
# Read CSV files
# -------------------------

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print()
print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)

print()
print("Train sentence_type counts:")
print(train_df["sentence_type"].value_counts())

print()
print("Val label counts:")
print(val_df["label"].value_counts())

print()
print("Test label counts:")
print(test_df["label"].value_counts())

# -------------------------
# Label mapping
# -------------------------

TRAIN_LABEL_MAP = {
    "statement": 0,
    "question": 1,
    "verb_start": 2,
    "exclamation": 3
}

VAL_TEST_LABEL_MAP = {
    "declarative": 0,
    "interrogative": 1,
    "imperative": 2,
    "exclamatory": 3
}

LABELS = [
    "declarative",
    "interrogative",
    "imperative",
    "exclamatory"
]

# -------------------------
# Collect audio files
# -------------------------

audio_dirs = [
    TRAIN_AUDIO_DIR,
    VAL_AUDIO_DIR,
    TEST_AUDIO_DIR
]

audio_files = []

print()
print("Counting audio files...")

for audio_dir in audio_dirs:
    count_here = 0

    print()
    print("Checking:", audio_dir)
    print("Folder exists:", os.path.exists(audio_dir))

    if os.path.exists(audio_dir):
        for root, dirs, files in os.walk(audio_dir):
            for f in files:
                if f.lower().endswith((".wav", ".mp3", ".aac", ".flac", ".ogg", ".m4a")):
                    audio_files.append(os.path.join(root, f))
                    count_here += 1

    print("Audio files here:", count_here)

print()
print("Total audio files found:", len(audio_files))

if len(audio_files) == 0:
    raise FileNotFoundError("No audio files found.")

# -------------------------
# Load Wav2Vec2 model
# -------------------------

MODEL_NAME = "facebook/wav2vec2-base-960h"
SR = 16000

# Important:
# hidden_states[0] = feature encoder output before transformer layers
# hidden_states[1] = transformer layer 1
# hidden_states[7] = transformer layer 7
LAYER_ID = 7

device = "cuda" if torch.cuda.is_available() else "cpu"

print()
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Using device:", device)
print("Using Wav2Vec2 hidden layer:", LAYER_ID)

print()
print("Loading Wav2Vec2 feature extractor...")
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)

print("Loading Wav2Vec2 model...")
wav_model = Wav2Vec2Model.from_pretrained(MODEL_NAME)

wav_model = wav_model.to(device)
wav_model.eval()

print("Wav2Vec2 model loaded.")

# -------------------------
# Create layer 7 embeddings
# -------------------------

def make_layer7_embedding(audio_path):
    audio, sr = librosa.load(audio_path, sr=SR)

    inputs = feature_extractor(
        audio,
        sampling_rate=SR,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = wav_model(
            **inputs,
            output_hidden_states=True
        )

    h = outputs.hidden_states[LAYER_ID]

    emb = h.mean(dim=1)
    emb = emb.squeeze(0).cpu().numpy()

    return emb


print()
print("Creating Wav2Vec2 layer 7 mean-pooled embeddings...")

saved = 0
skipped = 0
already_done = 0

for i, audio_path in enumerate(tqdm(audio_files), start=1):
    fname = os.path.splitext(os.path.basename(audio_path))[0]
    out_path = os.path.join(EMB_DIR, fname + ".npy")

    if os.path.exists(out_path):
        already_done += 1
        continue

    try:
        emb = make_layer7_embedding(audio_path)
        np.save(out_path, emb)
        saved += 1

    except Exception as e:
        skipped += 1
        print("[SKIP]", audio_path, "->", e)

    if i == 1 or i % 100 == 0:
        print(
            "Processed:",
            i,
            "out of",
            len(audio_files),
            "| saved:",
            saved,
            "| already existed:",
            already_done,
            "| skipped:",
            skipped
        )

print()
print("Layer 7 embedding creation done.")
print("Saved new embeddings:", saved)
print("Already existed:", already_done)
print("Skipped:", skipped)

emb_files = []

for root, dirs, files in os.walk(EMB_DIR):
    for f in files:
        if f.lower().endswith(".npy"):
            emb_files.append(os.path.join(root, f))

print("Total layer 7 embedding files:", len(emb_files))

if len(emb_files) > 0:
    sample = np.load(emb_files[0])
    print("Sample embedding shape:", sample.shape)
    print("Example embedding file:", emb_files[0])

del wav_model
torch.cuda.empty_cache()

# -------------------------
# Build embedding index
# -------------------------

def build_embedding_index(emb_dir):
    file_dict = {}
    duplicate_count = 0

    for root, dirs, files in os.walk(emb_dir):
        for f in files:
            if f.lower().endswith(".npy"):
                key = os.path.splitext(f)[0]
                path = os.path.join(root, f)

                if key in file_dict:
                    duplicate_count += 1

                file_dict[key] = path

    print()
    print("Indexed embeddings:", len(file_dict))
    print("Duplicate base names:", duplicate_count)

    if len(file_dict) == 0:
        raise FileNotFoundError("No embedding files found.")

    return file_dict

# -------------------------
# Load train/val/test embeddings
# -------------------------

def load_train_dataset(df, file_dict):
    X = []
    y = []
    missing = []
    bad_labels = []

    for idx, row in df.iterrows():
        fname = str(row["file_name"])
        label = str(row["sentence_type"]).strip().lower()

        key = os.path.splitext(os.path.basename(fname))[0]

        if label not in TRAIN_LABEL_MAP:
            bad_labels.append(label)
            continue

        if key in file_dict:
            emb = np.load(file_dict[key])
            X.append(emb)
            y.append(TRAIN_LABEL_MAP[label])
        else:
            missing.append(fname)

    X = np.array(X)
    y = np.array(y)

    print()
    print("Training CSV loaded")
    print("Rows:", len(df))
    print("Matched samples:", len(X))
    print("Missing samples:", len(missing))
    print("Bad labels:", len(bad_labels))

    if len(missing) > 0:
        print("First 20 missing:")
        for m in missing[:20]:
            print(m)

    if len(bad_labels) > 0:
        print("Unique bad labels:")
        print(sorted(list(set(bad_labels))))

    if len(X) == 0:
        raise ValueError("No training samples loaded.")

    return X, y


def load_eval_dataset(df, name, file_dict):
    X = []
    y = []
    missing = []
    bad_labels = []

    for idx, row in df.iterrows():
        fname = str(row["file_name"])
        label = str(row["label"]).strip().lower()

        key = os.path.splitext(os.path.basename(fname))[0]

        if label not in VAL_TEST_LABEL_MAP:
            bad_labels.append(label)
            continue

        if key in file_dict:
            emb = np.load(file_dict[key])
            X.append(emb)
            y.append(VAL_TEST_LABEL_MAP[label])
        else:
            missing.append(fname)

    X = np.array(X)
    y = np.array(y)

    print()
    print(name, "CSV loaded")
    print("Rows:", len(df))
    print("Matched samples:", len(X))
    print("Missing samples:", len(missing))
    print("Bad labels:", len(bad_labels))

    if len(missing) > 0:
        print("First 20 missing:")
        for m in missing[:20]:
            print(m)

    if len(bad_labels) > 0:
        print("Unique bad labels:")
        print(sorted(list(set(bad_labels))))

    if len(X) == 0:
        raise ValueError("No samples loaded for " + name)

    return X, y


file_dict = build_embedding_index(EMB_DIR)

X_train, y_train = load_train_dataset(train_df, file_dict)
X_val, y_val = load_eval_dataset(val_df, "Validation", file_dict)
X_test, y_test = load_eval_dataset(test_df, "Test", file_dict)

print()
print("Final shapes:")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

print()
print("Train class distribution:", np.bincount(y_train, minlength=4))
print("Val class distribution:", np.bincount(y_val, minlength=4))
print("Test class distribution:", np.bincount(y_test, minlength=4))

# -------------------------
# Dataset, projection model, loss
# -------------------------

class LabeledDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X).float()
        self.y = torch.tensor(y).long()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


class ProjectionHead(nn.Module):
    def __init__(self, dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(dim, 512),
            nn.ReLU(),
            nn.Linear(512, 128)
        )

    def forward(self, x):
        z = self.net(x)
        z = F.normalize(z, dim=1)
        return z


def supcon_loss(features, labels, temp=0.5):
    device = features.device

    labels = labels.view(-1, 1)

    mask = torch.eq(labels, labels.T).float().to(device)

    logits = torch.matmul(features, features.T) / temp

    self_mask = torch.eye(len(features), device=device)

    mask = mask * (1 - self_mask)

    exp_logits = torch.exp(logits) * (1 - self_mask)

    log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-9)

    mean_log_prob = (mask * log_prob).sum(1) / (mask.sum(1) + 1e-9)

    loss = -mean_log_prob.mean()

    return loss

# -------------------------
# Train projection model
# -------------------------

NUM_CLASSES = 4
EPOCHS = 80
BATCH_SIZE_LABELED = 64
BATCH_SIZE_PSEUDO = 128
LR = 1e-3
TEMP = 0.5

device = "cuda" if torch.cuda.is_available() else "cpu"

input_dim = X_train.shape[1]

print()
print("Training projection head on device:", device)

proj_model = ProjectionHead(input_dim).to(device)

optimizer = torch.optim.Adam(
    proj_model.parameters(),
    lr=LR
)

for epoch in range(EPOCHS):
    proj_model.train()

    labeled_loader = DataLoader(
        LabeledDataset(X_train, y_train),
        batch_size=BATCH_SIZE_LABELED,
        shuffle=True
    )

    total_loss = 0
    steps = 0

    for x, yb in labeled_loader:
        x = x.to(device)
        yb = yb.to(device)

        z = proj_model(x)

        loss = supcon_loss(z, yb, temp=TEMP)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        steps += 1

    proj_model.eval()

    with torch.no_grad():
        Z_train_all = proj_model(
            torch.tensor(X_train).float().to(device)
        ).cpu().numpy()

    pseudo = KMeans(
        n_clusters=NUM_CLASSES,
        n_init=10,
        random_state=42
    ).fit_predict(Z_train_all)

    pseudo_loader = DataLoader(
        LabeledDataset(X_train, pseudo),
        batch_size=BATCH_SIZE_PSEUDO,
        shuffle=True
    )

    proj_model.train()

    pseudo_loss_total = 0
    pseudo_steps = 0

    for x, yb in pseudo_loader:
        x = x.to(device)
        yb = yb.to(device)

        z = proj_model(x)

        loss = supcon_loss(z, yb, temp=TEMP)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        pseudo_loss_total += loss.item()
        pseudo_steps += 1

    avg_loss = total_loss / max(steps, 1)
    avg_pseudo_loss = pseudo_loss_total / max(pseudo_steps, 1)

    print(
        "Epoch",
        epoch + 1,
        "done | supervised loss:",
        round(avg_loss, 4),
        "| pseudo loss:",
        round(avg_pseudo_loss, 4)
    )

# -------------------------
# Logistic regression evaluation
# -------------------------

def project_embeddings(model, X):
    model.eval()

    with torch.no_grad():
        Z = model(
            torch.tensor(X).float().to(device)
        ).cpu().numpy()

    return Z


def evaluate(name, clf, X_proj, y_true):
    pred = clf.predict(X_proj)

    acc = accuracy_score(y_true, pred)
    prec = precision_score(y_true, pred, average="macro", zero_division=0)
    rec = recall_score(y_true, pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, pred, average="macro", zero_division=0)

    print()
    print("Results on", name)
    print("Accuracy:", acc)
    print("Precision macro:", prec)
    print("Recall macro:", rec)
    print("F1 macro:", f1)

    print()
    print("Classification Report:")
    print(
        classification_report(
            y_true,
            pred,
            target_names=LABELS,
            zero_division=0
        )
    )

    print()
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, pred))

    return pred, acc, f1


print()
print("Projecting embeddings...")

Z_train = project_embeddings(proj_model, X_train)
Z_val = project_embeddings(proj_model, X_val)
Z_test = project_embeddings(proj_model, X_test)

print("Projected train shape:", Z_train.shape)
print("Projected val shape:", Z_val.shape)
print("Projected test shape:", Z_test.shape)

print()
print("Training Logistic Regression classifier...")

clf = LogisticRegression(
    max_iter=3000,
    random_state=42,
    class_weight="balanced"
)

clf.fit(Z_train, y_train)

val_pred, val_acc, val_f1 = evaluate("validation set", clf, Z_val, y_val)
test_pred, test_acc, test_f1 = evaluate("test set", clf, Z_test, y_test)

# -------------------------
# Save outputs
# -------------------------

torch.save(
    proj_model.state_dict(),
    os.path.join(SAVE_DIR, "wav2vec2_layer7_projection_head.pt")
)

joblib.dump(
    clf,
    os.path.join(SAVE_DIR, "layer7_logistic_regression_classifier.joblib")
)

np.save(os.path.join(SAVE_DIR, "layer7_val_predictions.npy"), val_pred)
np.save(os.path.join(SAVE_DIR, "layer7_test_predictions.npy"), test_pred)

summary_path = os.path.join(SAVE_DIR, "layer7_run_summary.txt")

with open(summary_path, "w", encoding="utf-8") as f:
    f.write("Wav2Vec2 layer 7 speech-only sentence type classification\n")
    f.write("\n")
    f.write("Embedding layer used: hidden_states[7]\n")
    f.write("Pooling: mean over time\n")
    f.write("\n")
    f.write("Train mapping:\n")
    f.write("statement -> declarative\n")
    f.write("question -> interrogative\n")
    f.write("verb_start -> imperative\n")
    f.write("exclamation -> exclamatory\n")
    f.write("\n")
    f.write("X_train: " + str(X_train.shape) + "\n")
    f.write("X_val: " + str(X_val.shape) + "\n")
    f.write("X_test: " + str(X_test.shape) + "\n")
    f.write("\n")
    f.write("Validation accuracy: " + str(val_acc) + "\n")
    f.write("Validation macro F1: " + str(val_f1) + "\n")
    f.write("Test accuracy: " + str(test_acc) + "\n")
    f.write("Test macro F1: " + str(test_f1) + "\n")

print()
print("Saved files to:")
print(SAVE_DIR)

print()
print("Saved files:")
for f in os.listdir(SAVE_DIR):
    print(os.path.join(SAVE_DIR, f))

# -------------------------
# Create zip for download
# -------------------------

os.chdir("/kaggle/working")

ZIP_NAME = "wav2vec2_layer7_sentence_type_outputs.zip"
ZIP_BASE = "wav2vec2_layer7_sentence_type_outputs"

if os.path.exists(ZIP_NAME):
    os.remove(ZIP_NAME)

shutil.make_archive(
    ZIP_BASE,
    "zip",
    SAVE_DIR
)

print()
print("Zip created:")
print("/kaggle/working/" + ZIP_NAME)

print()
print("Zip size:")
!ls -lh /kaggle/working/wav2vec2_layer7_sentence_type_outputs.zip

display(FileLink(ZIP_NAME))

print()
print("All done.")

Installing required libraries...
DATASET_ROOT: /kaggle/input/datasets/ruhinbhattasali/iasnlp-dataset/dataset
TRAIN_AUDIO_DIR exists: True
VAL_AUDIO_DIR exists: True
TEST_AUDIO_DIR exists: True
TRAIN_CSV exists: True
VAL_CSV exists: True
TEST_CSV exists: True
EMB_DIR: /kaggle/working/wav2vec2_layer7_mean_pooled_embeddings
SAVE_DIR: /kaggle/working/wav2vec2_layer7_sentence_type_outputs

Train shape: (4000, 4)
Val shape: (202, 3)
Test shape: (202, 3)

Train sentence_type counts:
sentence_type
verb_start     1100
statement      1100
question       1100
exclamation     700
Name: count, dtype: int64

Val label counts:
label
declarative      53
exclamatory      51
interrogative    49
imperative       49
Name: count, dtype: int64

Test label counts:
label
declarative      53
exclamatory      51
imperative       50
interrogative    48
Name: count, dtype: int64

Counting audio files...

Checking: /kaggle/input/datasets/ruhinbhattasali/iasnlp-dataset/dataset/new_train_audio_batches
Folder exists:

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Wav2Vec2 model loaded.

Creating Wav2Vec2 layer 7 mean-pooled embeddings...


  0%|          | 3/4404 [00:00<02:41, 27.24it/s]

...

Processed: 4300 out of 4404 | saved: 4300 | already existed: 0 | skipped: 0


100%|██████████| 4404/4404 [01:56<00:00, 37.85it/s]


Processed: 4400 out of 4404 | saved: 4400 | already existed: 0 | skipped: 0

Layer 7 embedding creation done.
Saved new embeddings: 4404
Already existed: 0
Skipped: 0
Total layer 7 embedding files: 4404
Sample embedding shape: (768,)
Example embedding file: /kaggle/working/wav2vec2_layer7_mean_pooled_embeddings/4_0_d1305.npy

Indexed embeddings: 4404
Duplicate base names: 0

Training CSV loaded
Rows: 4000
Matched samples: 4000
Missing samples: 0
Bad labels: 0

Validation CSV loaded
Rows: 202
Matched samples: 202
Missing samples: 0
Bad labels: 0

Test CSV loaded
Rows: 202
Matched samples: 202
Missing samples: 0
Bad labels: 0

Final shapes:
X_train: (4000, 768) y_train: (4000,)
X_val: (202, 768) y_val: (202,)
X_test: (202, 768) y_test: (202,)

Train class distribution: [1100 1100 1100  700]
Val class distribution: [53 49 49 51]
Test class distribution: [53 48 50 51]

Training projection head on device: cuda
Epoch 1 done | supervised loss: 3.9589 | pseudo loss: 4.1029
Epoch 2 done | super

/kaggle/working/wav2vec2_layer7_sentence_type_outputs.zip


All done.


In [ ]:
import os
import sys
import subprocess

print("Installing required libraries...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers",
        "librosa",
        "soundfile",
        "scikit-learn",
        "tqdm",
        "joblib"
    ],
    check=True
)

import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import joblib
import shutil

from tqdm import tqdm
from IPython.display import FileLink, display
from transformers import AutoFeatureExtractor, Wav2Vec2Model
from torch.utils.data import Dataset, DataLoader

from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

DATASET_ROOT = "/kaggle/input/datasets/ruhinbhattasali/iasnlp-dataset/dataset"

TRAIN_AUDIO_DIR = os.path.join(DATASET_ROOT, "new_train_audio_batches")
VAL_AUDIO_DIR = os.path.join(DATASET_ROOT, "val200_audio_batches")
TEST_AUDIO_DIR = os.path.join(DATASET_ROOT, "test200_audio_batches")

TRAIN_CSV = os.path.join(DATASET_ROOT, "train_dataset_new.csv")
VAL_CSV = os.path.join(DATASET_ROOT, "val_dataset.csv")
TEST_CSV = os.path.join(DATASET_ROOT, "test_dataset.csv")

LAYER_ID = 8

EMB_DIR = "/kaggle/working/wav2vec2_layer8_mean_pooled_embeddings"
SAVE_DIR = "/kaggle/working/wav2vec2_layer8_sentence_type_outputs"

MODEL_NAME = "facebook/wav2vec2-base-960h"
SR = 16000

EPOCHS = 80
NUM_CLASSES = 4
BATCH_SIZE_LABELED = 64
BATCH_SIZE_PSEUDO = 128
LR = 1e-3
TEMP = 0.5

os.makedirs(EMB_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

print("DATASET_ROOT:", DATASET_ROOT)
print("TRAIN_AUDIO_DIR exists:", os.path.exists(TRAIN_AUDIO_DIR))
print("VAL_AUDIO_DIR exists:", os.path.exists(VAL_AUDIO_DIR))
print("TEST_AUDIO_DIR exists:", os.path.exists(TEST_AUDIO_DIR))
print("TRAIN_CSV exists:", os.path.exists(TRAIN_CSV))
print("VAL_CSV exists:", os.path.exists(VAL_CSV))
print("TEST_CSV exists:", os.path.exists(TEST_CSV))
print("EMB_DIR:", EMB_DIR)
print("SAVE_DIR:", SAVE_DIR)
print("Layer used:", LAYER_ID)

# -------------------------
# Read CSVs
# -------------------------

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print()
print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)

print()
print("Train sentence_type counts:")
print(train_df["sentence_type"].value_counts())

print()
print("Val label counts:")
print(val_df["label"].value_counts())

print()
print("Test label counts:")
print(test_df["label"].value_counts())

# -------------------------
# Label mapping
# -------------------------

TRAIN_LABEL_MAP = {
    "statement": 0,
    "question": 1,
    "verb_start": 2,
    "exclamation": 3
}

VAL_TEST_LABEL_MAP = {
    "declarative": 0,
    "interrogative": 1,
    "imperative": 2,
    "exclamatory": 3
}

LABELS = [
    "declarative",
    "interrogative",
    "imperative",
    "exclamatory"
]

# -------------------------
# Collect audio files
# -------------------------

audio_dirs = [
    TRAIN_AUDIO_DIR,
    VAL_AUDIO_DIR,
    TEST_AUDIO_DIR
]

audio_files = []

print()
print("Counting audio files...")

for audio_dir in audio_dirs:
    count_here = 0

    print()
    print("Checking:", audio_dir)
    print("Folder exists:", os.path.exists(audio_dir))

    if os.path.exists(audio_dir):
        for root, dirs, files in os.walk(audio_dir):
            for f in files:
                if f.lower().endswith((".wav", ".mp3", ".aac", ".flac", ".ogg", ".m4a")):
                    audio_files.append(os.path.join(root, f))
                    count_here += 1

    print("Audio files here:", count_here)

print()
print("Total audio files found:", len(audio_files))

if len(audio_files) == 0:
    raise FileNotFoundError("No audio files found.")

# -------------------------
# Load Wav2Vec2
# -------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"

print()
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Using device:", device)
print("Using Wav2Vec2 hidden layer:", LAYER_ID)

print()
print("Loading Wav2Vec2 feature extractor...")
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)

print("Loading Wav2Vec2 model...")
wav_model = Wav2Vec2Model.from_pretrained(MODEL_NAME)
wav_model = wav_model.to(device)
wav_model.eval()

print("Wav2Vec2 model loaded.")

# -------------------------
# Create layer 8 embeddings
# -------------------------

def make_layer_embedding(audio_path):
    audio, sr = librosa.load(audio_path, sr=SR)

    inputs = feature_extractor(
        audio,
        sampling_rate=SR,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = wav_model(
            **inputs,
            output_hidden_states=True
        )

    h = outputs.hidden_states[LAYER_ID]

    emb = h.mean(dim=1)
    emb = emb.squeeze(0).cpu().numpy()

    return emb


print()
print("Creating Wav2Vec2 layer 8 mean-pooled embeddings...")

saved = 0
skipped = 0
already_done = 0

for i, audio_path in enumerate(tqdm(audio_files), start=1):
    fname = os.path.splitext(os.path.basename(audio_path))[0]
    out_path = os.path.join(EMB_DIR, fname + ".npy")

    if os.path.exists(out_path):
        already_done += 1
        continue

    try:
        emb = make_layer_embedding(audio_path)
        np.save(out_path, emb)
        saved += 1

    except Exception as e:
        skipped += 1
        print("[SKIP]", audio_path, "->", e)

    if i == 1 or i % 100 == 0:
        print(
            "Processed:",
            i,
            "out of",
            len(audio_files),
            "| saved:",
            saved,
            "| already existed:",
            already_done,
            "| skipped:",
            skipped
        )

print()
print("Layer 8 embedding creation done.")
print("Saved new embeddings:", saved)
print("Already existed:", already_done)
print("Skipped:", skipped)

emb_files = []

for root, dirs, files in os.walk(EMB_DIR):
    for f in files:
        if f.lower().endswith(".npy"):
            emb_files.append(os.path.join(root, f))

print("Total layer 8 embedding files:", len(emb_files))

if len(emb_files) > 0:
    sample = np.load(emb_files[0])
    print("Sample embedding shape:", sample.shape)
    print("Example embedding file:", emb_files[0])

del wav_model
torch.cuda.empty_cache()

# -------------------------
# Build embedding index
# -------------------------

def build_embedding_index(emb_dir):
    file_dict = {}
    duplicate_count = 0

    for root, dirs, files in os.walk(emb_dir):
        for f in files:
            if f.lower().endswith(".npy"):
                key = os.path.splitext(f)[0]
                path = os.path.join(root, f)

                if key in file_dict:
                    duplicate_count += 1

                file_dict[key] = path

    print()
    print("Indexed embeddings:", len(file_dict))
    print("Duplicate base names:", duplicate_count)

    if len(file_dict) == 0:
        raise FileNotFoundError("No embedding files found.")

    return file_dict


# -------------------------
# Load train/val/test data
# -------------------------

def load_train_dataset(df, file_dict):
    X = []
    y = []
    missing = []
    bad_labels = []

    for idx, row in df.iterrows():
        fname = str(row["file_name"])
        label = str(row["sentence_type"]).strip().lower()

        key = os.path.splitext(os.path.basename(fname))[0]

        if label not in TRAIN_LABEL_MAP:
            bad_labels.append(label)
            continue

        if key in file_dict:
            emb = np.load(file_dict[key])
            X.append(emb)
            y.append(TRAIN_LABEL_MAP[label])
        else:
            missing.append(fname)

    X = np.array(X)
    y = np.array(y)

    print()
    print("Training CSV loaded")
    print("Rows:", len(df))
    print("Matched samples:", len(X))
    print("Missing samples:", len(missing))
    print("Bad labels:", len(bad_labels))

    if len(missing) > 0:
        print("First 20 missing:")
        for m in missing[:20]:
            print(m)

    if len(bad_labels) > 0:
        print("Unique bad labels:")
        print(sorted(list(set(bad_labels))))

    if len(X) == 0:
        raise ValueError("No training samples loaded.")

    return X, y


def load_eval_dataset(df, name, file_dict):
    X = []
    y = []
    missing = []
    bad_labels = []

    for idx, row in df.iterrows():
        fname = str(row["file_name"])
        label = str(row["label"]).strip().lower()

        key = os.path.splitext(os.path.basename(fname))[0]

        if label not in VAL_TEST_LABEL_MAP:
            bad_labels.append(label)
            continue

        if key in file_dict:
            emb = np.load(file_dict[key])
            X.append(emb)
            y.append(VAL_TEST_LABEL_MAP[label])
        else:
            missing.append(fname)

    X = np.array(X)
    y = np.array(y)

    print()
    print(name, "CSV loaded")
    print("Rows:", len(df))
    print("Matched samples:", len(X))
    print("Missing samples:", len(missing))
    print("Bad labels:", len(bad_labels))

    if len(missing) > 0:
        print("First 20 missing:")
        for m in missing[:20]:
            print(m)

    if len(bad_labels) > 0:
        print("Unique bad labels:")
        print(sorted(list(set(bad_labels))))

    if len(X) == 0:
        raise ValueError("No samples loaded for " + name)

    return X, y


file_dict = build_embedding_index(EMB_DIR)

X_train, y_train = load_train_dataset(train_df, file_dict)
X_val, y_val = load_eval_dataset(val_df, "Validation", file_dict)
X_test, y_test = load_eval_dataset(test_df, "Test", file_dict)

print()
print("Final shapes:")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

print()
print("Train class distribution:", np.bincount(y_train, minlength=4))
print("Val class distribution:", np.bincount(y_val, minlength=4))
print("Test class distribution:", np.bincount(y_test, minlength=4))

# -------------------------
# Dataset, model, loss
# -------------------------

class LabeledDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X).float()
        self.y = torch.tensor(y).long()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


class ProjectionHead(nn.Module):
    def __init__(self, dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(dim, 512),
            nn.ReLU(),
            nn.Linear(512, 128)
        )

    def forward(self, x):
        z = self.net(x)
        z = F.normalize(z, dim=1)
        return z


def supcon_loss(features, labels, temp=0.5):
    device = features.device

    labels = labels.view(-1, 1)

    mask = torch.eq(labels, labels.T).float().to(device)

    logits = torch.matmul(features, features.T) / temp

    self_mask = torch.eye(len(features), device=device)

    mask = mask * (1 - self_mask)

    exp_logits = torch.exp(logits) * (1 - self_mask)

    log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-9)

    mean_log_prob = (mask * log_prob).sum(1) / (mask.sum(1) + 1e-9)

    loss = -mean_log_prob.mean()

    return loss

# -------------------------
# Train projection model
# -------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"

input_dim = X_train.shape[1]

print()
print("Training projection head on device:", device)

proj_model = ProjectionHead(input_dim).to(device)

optimizer = torch.optim.Adam(
    proj_model.parameters(),
    lr=LR
)

for epoch in range(EPOCHS):
    proj_model.train()

    labeled_loader = DataLoader(
        LabeledDataset(X_train, y_train),
        batch_size=BATCH_SIZE_LABELED,
        shuffle=True
    )

    total_loss = 0
    steps = 0

    for x, yb in labeled_loader:
        x = x.to(device)
        yb = yb.to(device)

        z = proj_model(x)

        loss = supcon_loss(z, yb, temp=TEMP)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        steps += 1

    proj_model.eval()

    with torch.no_grad():
        Z_train_all = proj_model(
            torch.tensor(X_train).float().to(device)
        ).cpu().numpy()

    pseudo = KMeans(
        n_clusters=NUM_CLASSES,
        n_init=10,
        random_state=42
    ).fit_predict(Z_train_all)

    pseudo_loader = DataLoader(
        LabeledDataset(X_train, pseudo),
        batch_size=BATCH_SIZE_PSEUDO,
        shuffle=True
    )

    proj_model.train()

    pseudo_loss_total = 0
    pseudo_steps = 0

    for x, yb in pseudo_loader:
        x = x.to(device)
        yb = yb.to(device)

        z = proj_model(x)

        loss = supcon_loss(z, yb, temp=TEMP)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        pseudo_loss_total += loss.item()
        pseudo_steps += 1

    avg_loss = total_loss / max(steps, 1)
    avg_pseudo_loss = pseudo_loss_total / max(pseudo_steps, 1)

    print(
        "Epoch",
        epoch + 1,
        "done | supervised loss:",
        round(avg_loss, 4),
        "| pseudo loss:",
        round(avg_pseudo_loss, 4)
    )

# -------------------------
# Logistic regression evaluation
# -------------------------

def project_embeddings(model, X):
    model.eval()

    with torch.no_grad():
        Z = model(
            torch.tensor(X).float().to(device)
        ).cpu().numpy()

    return Z


def evaluate(name, clf, X_proj, y_true):
    pred = clf.predict(X_proj)

    acc = accuracy_score(y_true, pred)
    prec = precision_score(y_true, pred, average="macro", zero_division=0)
    rec = recall_score(y_true, pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, pred, average="macro", zero_division=0)

    print()
    print("Results on", name)
    print("Accuracy:", acc)
    print("Precision macro:", prec)
    print("Recall macro:", rec)
    print("F1 macro:", f1)

    print()
    print("Classification Report:")
    print(
        classification_report(
            y_true,
            pred,
            target_names=LABELS,
            zero_division=0
        )
    )

    print()
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, pred))

    return pred, acc, f1


print()
print("Projecting embeddings...")

Z_train = project_embeddings(proj_model, X_train)
Z_val = project_embeddings(proj_model, X_val)
Z_test = project_embeddings(proj_model, X_test)

print("Projected train shape:", Z_train.shape)
print("Projected val shape:", Z_val.shape)
print("Projected test shape:", Z_test.shape)

print()
print("Training Logistic Regression classifier...")

clf = LogisticRegression(
    max_iter=3000,
    random_state=42,
    class_weight="balanced"
)

clf.fit(Z_train, y_train)

val_pred, val_acc, val_f1 = evaluate("validation set", clf, Z_val, y_val)
test_pred, test_acc, test_f1 = evaluate("test set", clf, Z_test, y_test)

# -------------------------
# Save outputs
# -------------------------

torch.save(
    proj_model.state_dict(),
    os.path.join(SAVE_DIR, "wav2vec2_layer8_projection_head.pt")
)

joblib.dump(
    clf,
    os.path.join(SAVE_DIR, "layer8_logistic_regression_classifier.joblib")
)

np.save(os.path.join(SAVE_DIR, "layer8_val_predictions.npy"), val_pred)
np.save(os.path.join(SAVE_DIR, "layer8_test_predictions.npy"), test_pred)

summary_path = os.path.join(SAVE_DIR, "layer8_run_summary.txt")

with open(summary_path, "w", encoding="utf-8") as f:
    f.write("Wav2Vec2 layer 8 speech-only sentence type classification\n")
    f.write("\n")
    f.write("Embedding layer used: hidden_states[8]\n")
    f.write("Pooling: mean over time\n")
    f.write("\n")
    f.write("Train mapping:\n")
    f.write("statement -> declarative\n")
    f.write("question -> interrogative\n")
    f.write("verb_start -> imperative\n")
    f.write("exclamation -> exclamatory\n")
    f.write("\n")
    f.write("X_train: " + str(X_train.shape) + "\n")
    f.write("X_val: " + str(X_val.shape) + "\n")
    f.write("X_test: " + str(X_test.shape) + "\n")
    f.write("\n")
    f.write("Validation accuracy: " + str(val_acc) + "\n")
    f.write("Validation macro F1: " + str(val_f1) + "\n")
    f.write("Test accuracy: " + str(test_acc) + "\n")
    f.write("Test macro F1: " + str(test_f1) + "\n")

print()
print("Saved files to:")
print(SAVE_DIR)

print()
print("Saved files:")
for f in os.listdir(SAVE_DIR):
    print(os.path.join(SAVE_DIR, f))

# -------------------------
# Create zip for download
# -------------------------

os.chdir("/kaggle/working")

ZIP_NAME = "wav2vec2_layer8_sentence_type_outputs.zip"
ZIP_BASE = "wav2vec2_layer8_sentence_type_outputs"

if os.path.exists(ZIP_NAME):
    os.remove(ZIP_NAME)

shutil.make_archive(
    ZIP_BASE,
    "zip",
    SAVE_DIR
)

print()
print("Zip created:")
print("/kaggle/working/" + ZIP_NAME)

print()
print("Zip size:")
!ls -lh /kaggle/working/wav2vec2_layer8_sentence_type_outputs.zip

display(FileLink(ZIP_NAME))

print()
print("All done.")

Installing required libraries...
DATASET_ROOT: /kaggle/input/datasets/ruhinbhattasali/iasnlp-dataset/dataset
TRAIN_AUDIO_DIR exists: True
VAL_AUDIO_DIR exists: True
TEST_AUDIO_DIR exists: True
TRAIN_CSV exists: True
VAL_CSV exists: True
TEST_CSV exists: True
EMB_DIR: /kaggle/working/wav2vec2_layer8_mean_pooled_embeddings
SAVE_DIR: /kaggle/working/wav2vec2_layer8_sentence_type_outputs
Layer used: 8

Train shape: (4000, 4)
Val shape: (202, 3)
Test shape: (202, 3)

Train sentence_type counts:
sentence_type
verb_start     1100
statement      1100
question       1100
exclamation     700
Name: count, dtype: int64

Val label counts:
label
declarative      53
exclamatory      51
interrogative    49
imperative       49
Name: count, dtype: int64

Test label counts:
label
declarative      53
exclamatory      51
imperative       50
interrogative    48
Name: count, dtype: int64

Counting audio files...

Checking: /kaggle/input/datasets/ruhinbhattasali/iasnlp-dataset/dataset/new_train_audio_batches


Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Wav2Vec2 model loaded.

Creating Wav2Vec2 layer 8 mean-pooled embeddings...


  0%|          | 3/4404 [00:00<02:44, 26.77it/s]

...

100%|██████████| 4404/4404 [01:55<00:00, 38.11it/s]

Processed: 4400 out of 4404 | saved: 4400 | already existed: 0 | skipped: 0

Layer 8 embedding creation done.
Saved new embeddings: 4404
Already existed: 0
Skipped: 0
Total layer 8 embedding files: 4404
Sample embedding shape: (768,)
Example embedding file: /kaggle/working/wav2vec2_layer8_mean_pooled_embeddings/4_0_d1305.npy

Indexed embeddings: 4404
Duplicate base names: 0



Training CSV loaded
Rows: 4000
Matched samples: 4000
Missing samples: 0
Bad labels: 0

Validation CSV loaded
Rows: 202
Matched samples: 202
Missing samples: 0
Bad labels: 0

Test CSV loaded
Rows: 202
Matched samples: 202
Missing samples: 0
Bad labels: 0

Final shapes:
X_train: (4000, 768) y_train: (4000,)
X_val: (202, 768) y_val: (202,)
X_test: (202, 768) y_test: (202,)

Train class distribution: [1100 1100 1100  700]
Val class distribution: [53 49 49 51]
Test class distribution: [53 48 50 51]

Training projection head on device: cuda
Epoch 1 done | supervised loss: 3.9808 | pseudo loss: 4.0716
Epoch 2 done | supervised loss: 3.8387 | pseudo loss: 4.0603
Epoch 3 done | supervised loss: 3.7491 | pseudo loss: 4.0262
Epoch 4 done | supervised loss: 3.6871 | pseudo loss: 4.0004
Epoch 5 done | supervised loss: 3.6067 | pseudo loss: 3.9572
Epoch 6 done | supervised loss: 3.5195 | pseudo loss: 3.921
Epoch 7 done | supervised loss: 3.4817 | pseudo loss: 3.9099
Epoch 8 done | supervised loss: 

/kaggle/working/wav2vec2_layer8_sentence_type_outputs.zip


All done.
